# 02 - Preprocessing & EDA
Continues from notebook 01. Performs:
- Stopword removal
- Stemming
- Sentiment labeling
- Exploratory Data Analysis (pie chart & bar chart)

In [ ]:
!pip install google-play-scraper PySastrawi

In [ ]:
import pandas as pd
import re
import ast
import time
import nltk
import matplotlib.pyplot as plt
from collections import Counter
from google_play_scraper import reviews_all, Sort
from nltk.tokenize import word_tokenize
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

nltk.download('punkt')
nltk.download('punkt_tab')

## 1. Load or Scrape Data

In [ ]:
# Option A: load from CSV if already scraped
# df_raw = pd.read_csv('mtix_raw.csv')
# df_raw['clean_tokens'] = df_raw['clean_tokens'].apply(ast.literal_eval)

# Option B: scrape fresh
def scrape_reviews(app_id, langs, country='id'):
 all_dfs = []
 for lang in langs:
 print(f'Scraping lang={lang}...')
 start = time.time()
 result = reviews_all(app_id, lang=lang, country=country, sort=Sort.NEWEST, sleep_milliseconds=0)
 elapsed = time.time() - start
 print(f'Done: {len(result)} reviews in {elapsed:.2f}s')
 df_lang = pd.DataFrame(result)
 df_lang['lang'] = lang
 all_dfs.append(df_lang)
 df_all = pd.concat(all_dfs, ignore_index=True)
 return df_all

df_raw = scrape_reviews('lds.cinema21', langs=['id', 'en'])
df_raw = df_raw[['content', 'score']].dropna(subset=['content'])
df_raw = df_raw[df_raw['content'].str.strip() != '']
print(df_raw.shape)
df_raw.head()

## 2. Tokenization & Punctuation Removal

In [ ]:
df_raw['tokenized'] = df_raw['content'].apply(word_tokenize)
df_raw['lower_tokens'] = df_raw['tokenized'].apply(lambda t: [x.lower() for x in t])
df_raw['clean_tokens'] = df_raw['lower_tokens'].apply(
 lambda t: [x for x in t if re.match(r'[a-zA-Z0-9]', x)]
)
df_raw[['content', 'clean_tokens']].head()

## 3. Stopword Removal

In [ ]:
stop_factory = StopWordRemoverFactory()
stopwords = stop_factory.get_stop_words()

df_raw['stopword_removed'] = df_raw['clean_tokens'].apply(
 lambda tokens: [t for t in tokens if t not in stopwords]
)
df_raw[['clean_tokens', 'stopword_removed']].head()

## 4. Stemming

In [ ]:
factory = StemmerFactory()
stemmer = factory.create_stemmer()

df_raw['stemmed'] = df_raw['stopword_removed'].apply(
 lambda tokens: [stemmer.stem(t) for t in tokens]
)
df_raw[['stopword_removed', 'stemmed']].head()

## 5. Sentiment Labeling

In [ ]:
def label_sentiment(score):
 if score >= 4:
 return 1 # Positive
 elif score <= 2:
 return 0 # Negative
 else:
 return None # Neutral - excluded

df_raw['label'] = df_raw['score'].apply(label_sentiment)
df_raw.dropna(subset=['label'], inplace=True)
df_raw['label'] = df_raw['label'].astype(int)

# Filter empty stemmed tokens
df_raw = df_raw[df_raw['stemmed'].map(len) > 0]

df_final = df_raw[['content', 'clean_tokens', 'stemmed', 'label']].copy()
print(f'Total data: {len(df_final)}')
print(df_final['label'].value_counts())
df_final.head()

## 6. Save Preprocessed Data

In [ ]:
df_final.to_csv('mtix_preprocessed.csv', index=False)
print('Preprocessed data saved.')

## 7. EDA - Sentiment Distribution (Pie Chart)

In [ ]:
label_counts = df_final['label'].value_counts()
labels = ['Positive' if i == 1 else 'Negative' for i in label_counts.index]
colors = ['steelblue', 'tomato']

plt.figure(figsize=(6, 6))
plt.pie(
 label_counts.values,
 labels=labels,
 colors=colors,
 autopct='%1.1f%%',
 startangle=90,
 wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
plt.title('Sentiment Distribution of M-Tix Reviews', fontsize=14)
plt.tight_layout()
plt.show()

## 8. EDA - Top 20 Most Frequent Words (Bar Chart)

In [ ]:
all_words = [word for tokens in df_final['stemmed'] for word in tokens]
word_freq = Counter(all_words).most_common(20)

words = [w[0] for w in word_freq]
freqs = [w[1] for w in word_freq]

plt.figure(figsize=(10, 7))
plt.barh(words[::-1], freqs[::-1], color='steelblue', edgecolor='black')
plt.title('Top 20 Most Frequent Words in M-Tix Reviews', fontsize=14)
plt.xlabel('Frequency')
plt.ylabel('Word')
for i, v in enumerate(freqs[::-1]):
 plt.text(v + 10, i, str(v), va='center', fontsize=9)
plt.tight_layout()
plt.show()